# Notebook 04: Speech Recognition (ASR)

**Time:** 25 minutes  
**Prerequisites:** Notebook 03 complete  
**Goal:** Transcribe audio using Whisper and faster-whisper, compare speed and accuracy

This notebook will:
1. Transcribe audio with OpenAI Whisper (baseline)
2. Use faster-whisper for 4x speedup with CTranslate2
3. Compare model sizes and their speed/accuracy trade-offs
4. Discuss real-world ASR for pretraining data (podcasts, lectures, YouTube)

> **Why this matters:** Audio is a massive untapped data source for LLM pretraining. Podcasts, lectures, and YouTube videos contain billions of tokens of high-quality spoken content. ASR converts this audio to text. Whisper v3 turbo (2025) processes audio 216x faster than real-time.

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

import src.audio_utils
importlib.reload(src.audio_utils)
from src.audio_utils import (
    transcribe_with_faster_whisper,
)

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 04")

✓ Gemini client initialized (gemini-3-flash-preview)
Setup complete -- ready for Notebook 04


---

## Part 1: OpenAI Whisper (Baseline)

Whisper is OpenAI's open-source ASR model. It supports 100+ languages and multiple model sizes:

| Model | Params | Speed | Best For |
|-------|--------|-------|----------|
| tiny | 39M | Fastest | Quick testing |
| base | 74M | Fast | Good default |
| small | 244M | Medium | Better accuracy |
| medium | 769M | Slow | High accuracy |
| large-v3 | 1.5B | Slowest | Best accuracy |
| turbo | 809M | 6x faster than large | Best speed/accuracy (2025) |

In [10]:
print("=" * 65)
print("Experiment 1: OpenAI Whisper Transcription")
print("=" * 65)
print()

# Find test audio
test_audio = os.path.join(parent_dir, 'test_data', 'audio', 'sample-1.mp3')

if not os.path.exists(test_audio):
    # Try Class3 test data
    class3_audio = os.path.join(parent_dir, '..', 'MLE_in_Gen_AI-Course', 'Class3', 'test_data', 'audio', 'sample-1.mp3')
    if os.path.exists(class3_audio):
        test_audio = class3_audio
        print(test_audio)
    else:
        print("No test audio found.")
        print("Place an audio file at: test_data/audio/sample-1.mp3")
        test_audio = None
if test_audio:
    try:
        from src.audio_utils import transcribe_with_whisper
        whisper_result = transcribe_with_whisper(test_audio, model_size="base")
    except ImportError as e:
        print(f"Whisper not installed: {e}")
        print("Install with: pip install openai-whisper")
        print("Continuing with faster-whisper in Part 2...")

Experiment 1: OpenAI Whisper Transcription

WHISPER ASR (model: base)
Audio: /home/chris/Homework3-Submission/test_data/audio/sample-1.mp3


/home/chris/Homework3-Submission/.venv/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  Language: en
  Segments: 3
  Transcribed in 10.6s
  Text: We pay for that, they didn't, you know, that some people said it's on. Cren the Benzo instead of caught in a musical or just not to worry about your interest in this journey. And that's the......


---

## Part 2: faster-whisper (4x Speedup)

**faster-whisper** uses CTranslate2 to run Whisper models 4x faster with lower memory. It's the recommended backend for production ASR.

Key features:
- Supports all Whisper model sizes including **turbo**
- Batched inference for processing multiple files
- INT8 quantization for CPU efficiency
- Word-level timestamps

In [12]:
print("=" * 65)
print("Experiment 2: faster-whisper Transcription")
print("=" * 65)
print()

if test_audio:
    fw_result = transcribe_with_faster_whisper(
        test_audio,
        model_size="base",
        device="cpu",
        compute_type="int8"
    )
    
    print("\nTimestamped segments:")
    for seg in fw_result['segments']:
        print(f"  [{seg['start']:.1f}s - {seg['end']:.1f}s] {seg['text']}")
else:
    print("No test audio available.")

Experiment 2: faster-whisper Transcription

FASTER-WHISPER ASR (model: base, cpu/int8)
Audio: /home/chris/Homework3-Submission/test_data/audio/sample-1.mp3
  Language: en (prob: 0.608)
  Segments: 3
  Transcribed in 2.0s
  Text: So we pay for that, they didn't, you know, there are some people who said it's on. Cren the Benzo instead of court and a musical or there's not a story of what you're interested in. It's just turning ...

Timestamped segments:
  [0.0s - 4.0s] So we pay for that, they didn't, you know, there are some people who said it's on.
  [4.0s - 8.0s] Cren the Benzo instead of court and a musical or there's not a story of what you're interested in.
  [8.0s - 10.0s] It's just turning child, that's the...


In [20]:
print("=" * 65)
print("Experiment 3: Model Size Comparison")
print("=" * 65)
print()

if test_audio:
    model_sizes = ["tiny", "base", "small"]
    results = {}
    
    for size in model_sizes:
        print(f"\n--- Testing model: {size} ---")
        result = transcribe_with_faster_whisper(
            test_audio, model_size=size, device="cpu", compute_type="int8"
        )
        results[size] = result
    
    print("\n\n--- Speed Comparison ---")
    for size, res in results.items():
        print(f"  {size:>8}: {res['elapsed_seconds']:.2f}s")
    
    print("\n--- Transcription Comparison ---")
    for size, res in results.items():
        print(f"  {size:>8}: {res['text'][:100]}...")
else:
    print("No test audio available.")

Experiment 3: Model Size Comparison


--- Testing model: tiny ---
FASTER-WHISPER ASR (model: tiny, cpu/int8)
Audio: /home/chris/Homework3-Submission/test_data/audio/sample-1.mp3
  Language: en (prob: 0.846)
  Segments: 3
  Transcribed in 1.6s
  Text: the way he pays a bat, they didn't, you know, that he's some people sector son. Cren the benzo instead of court and a musical or just not hurry up which one just understanding child, that's me....

--- Testing model: base ---
FASTER-WHISPER ASR (model: base, cpu/int8)
Audio: /home/chris/Homework3-Submission/test_data/audio/sample-1.mp3
  Language: en (prob: 0.608)
  Segments: 3
  Transcribed in 2.0s
  Text: So we pay for that, they didn't, you know, there are some people who said it's on. Cren the Benzo instead of court and a musical or there's not a story of what you're interested in. It's just turning ...

--- Testing model: small ---
FASTER-WHISPER ASR (model: small, cpu/int8)
Audio: /home/chris/Homework3-Submission/test_data/audio/samp

vocabulary.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/484M [00:00<?, ?B/s]

  Language: en (prob: 0.848)
  Segments: 5
  Transcribed in 9.9s
  Text: The way he pays for that, they didn't, you know, that he's somebody who's like his son. Karen Davenso instead of Corten and Musico, I'll just not tell you what you're and she'll send this turning chil...


--- Speed Comparison ---
      tiny: 1.59s
      base: 2.04s
     small: 9.86s

--- Transcription Comparison ---
      tiny: the way he pays a bat, they didn't, you know, that he's some people sector son. Cren the benzo inste...
      base: So we pay for that, they didn't, you know, there are some people who said it's on. Cren the Benzo in...
     small: The way he pays for that, they didn't, you know, that he's somebody who's like his son. Karen Davens...


In [21]:
# TODO 1: Analyze ASR for pretraining data
#
# Use the LLM to discuss how ASR fits into a pretraining pipeline.

print("=" * 65)
print("TODO 1: ASR in Pretraining Pipelines")
print("=" * 65)
print()

start = time.time()
response = client.generate(
    prompt="""Explain how Automatic Speech Recognition (ASR) is used to build LLM pretraining datasets.

Cover:
1. What audio sources are typically transcribed (podcasts, lectures, YouTube)?
2. What quality issues arise from ASR transcriptions vs written text?
3. How do modern models like Whisper v3 turbo (2025) compare to earlier ASR?
4. What post-processing is needed before ASR text goes into a training dataset?

Be practical and give specific examples.""",
    system="You are an expert in building LLM pretraining data pipelines.",
    max_tokens=2500,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")


TODO 1: ASR in Pretraining Pipelines

⚡ Generating with Gemini...
Response in 15.8s
Model: gemini-3-flash-preview
Tokens: 121 in, 1343 out
Stop reason: FinishReason.STOP
As high-quality "human-written" text on the open web becomes a scarce resource, audio has become the new frontier for LLM pretraining. Audio data provides a massive repository of conversational logic, specialized knowledge, and cultural nuances that are often absent from formal books or Wikipedia.

Here is how modern LLM pipelines leverage Automatic Speech Recognition (ASR) to turn audio into training tokens.

---

### 1. Typical Audio Sources for LLM Data
Data engineers target sources that provide high "information density."

*   **YouTube (The "Holy Grail"):** YouTube contains trillions of tokens. Pipelines target educational channels (e.g., PBS Space Time), DIY tutorials, and debates. The diversity of accents and topics is unmatched.
*   **Podcasts:** These are highly valued because they contain long-form, multi-tur

In [22]:

todo1_reflection = """


- How does ASR transcription quality compare to web-scraped text?
 ASR transcriptions can contain errors such as misheard words, incorrect punctuation, and lack of formatting. 
 Web-scraped text is typically cleaner but may still have issues like HTML artifacts or OCR errors.

- Which model size would you choose for transcribing 1000 hours of podcasts?
 I would likely choose the "small" model for a balance of speed and accuracy. 
 The "tiny" model may be too inaccurate, while the "base" model may be too slow for large volumes.

- What are the ethical considerations of transcribing public audio content?
Well, there are several ethical considerations to keep in mind when transcribing public audio content:
1. **Privacy**: Even if the audio is publicly available, it may contain sensitive information about individuals. Transcribing and using this data without consent can lead to privacy violations.
2. **Copyright**: Many audio sources, such as podcasts and YouTube videos, are
protected by copyright. Transcribing and using this content for training without permission may infringe on intellectual property rights.
3. **Bias**: ASR models may have biases that affect transcription quality for certain accents
or dialects. This can lead to underrepresentation of certain groups in the training data, which may perpetuate biases in the resulting LLM.
4. **Misinformation**: If the audio content contains misinformation or harmful content,
transcribing and including it in training data can lead to the LLM learning and reproducing that misinformation.
5. **Transparency**: It's important to be transparent about the sources of training data and the
methods used for transcription, so that users of the LLM can understand the potential limitations and biases in the model.



"""

print()
print(todo1_reflection)





- How does ASR transcription quality compare to web-scraped text?
 ASR transcriptions can contain errors such as misheard words, incorrect punctuation, and lack of formatting. 
 Web-scraped text is typically cleaner but may still have issues like HTML artifacts or OCR errors.

- Which model size would you choose for transcribing 1000 hours of podcasts?
 I would likely choose the "small" model for a balance of speed and accuracy. 
 The "tiny" model may be too inaccurate, while the "base" model may be too slow for large volumes.

- What are the ethical considerations of transcribing public audio content?
Well, there are several ethical considerations to keep in mind when transcribing public audio content:
1. **Privacy**: Even if the audio is publicly available, it may contain sensitive information about individuals. Transcribing and using this data without consent can lead to privacy violations.
2. **Copyright**: Many audio sources, such as podcasts and YouTube videos, are
protected 

In [23]:
# TODO 2: Transcribe your own audio (optional)
#
# Record a 15-30 second audio clip or find one online.
# Transcribe it and evaluate the quality.

print("=" * 65)
print("TODO 2: Custom Audio Transcription (Optional)")
print("=" * 65)
print()

# my_audio = "path/to/your/audio.mp3"
# my_result = transcribe_with_faster_whisper(my_audio, model_size="small")

todo2_reflection = """


- What audio did you transcribe?
    I transcribed sample-1.mps in test data

- How accurate was the transcription?
   THe transcription was fairly accurate, capturing the main content of the audio. 
   However, there were some errors . 


- What would you change for better results (model size, preprocessing)?

    I would experiment with the "small" model for better accuracy, as the "tiny" model may have been too error-prone. 
    Additionally, I would consider preprocessing the audio to enhance clarity, such as noise reduction or normalization, which could improve transcription quality. 


"""

print(todo2_reflection)

TODO 2: Custom Audio Transcription (Optional)




- What audio did you transcribe?
    I transcribed sample-1.mps in test data

- How accurate was the transcription?
   THe transcription was fairly accurate, capturing the main content of the audio. 
   However, there were some errors . 


- What would you change for better results (model size, preprocessing)?

    I would experiment with the "small" model for better accuracy, as the "tiny" model may have been too error-prone. 
    Additionally, I would consider preprocessing the audio to enhance clarity, such as noise reduction or normalization, which could improve transcription quality. 





---

## Summary & Reflection

In [24]:
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'

full_reflection = f"""
### Part 1 - ASR in Pretraining

{_todo1}

---

### Part 2 - Custom Transcription

{_todo2}
"""

reflection_file = append_to_reflection(
    notebook="04",
    section_title="Speech Recognition (ASR)",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     1
Total input tokens:  121
Total output tokens: 1,343
Total cost:          $0.0205

Last 1 calls:
  1. [17:56:03] 3 -- 121in/1343out -- $0.0205


## Notebook 04 Complete!

**What you accomplished:**
- Transcribed audio with OpenAI Whisper and faster-whisper
- Compared model sizes for speed vs accuracy
- Explored how ASR fits into pretraining pipelines

**Key concepts:**
- faster-whisper provides 4x speedup with CTranslate2 backend
- Whisper v3 turbo (2025) offers best speed/accuracy trade-off
- ASR text needs post-processing before use in pretraining

**Next:** Open **Notebook 05: Data Cleaning Pipeline**